In [43]:
import os
import shutil
from pathlib import Path
from lxml import etree
from lxml.etree import ElementTree, Element

DIR_DATA_SAMPLE = Path(Path.cwd().parent, "data")
DIR_SEMPER_SAMPLE = Path(DIR_DATA_SAMPLE, "semper")
DIR_HWGW_SAMPLE = Path(DIR_DATA_SAMPLE, "hwgw")
DIR_OUT = Path(Path.cwd().parent, "dts_out")

list(DIR_SEMPER_SAMPLE.glob("*.*"))
list(DIR_HWGW_SAMPLE.glob("*.*"))

NS = {"tei": "http://www.tei-c.org/ns/1.0"}

In [ ]:
from copy import deepcopy


def parse_xml(path: Path | str) -> ElementTree:
    parser = etree.XMLParser(remove_blank_text=True, load_dtd=True)
    return etree.parse(str(path), parser)


def print_etree(etre: ElementTree):
    print(etree.tostring(etre, pretty_print=True, encoding="unicode"))


def write_xml_with_schema(tree: ElementTree, output_path: Path) -> None:
    root = deepcopy(tree.getroot())
    pi = etree.ProcessingInstruction("oxygen", 'RNGSchema="schema.rng" type="xml"')
    root.addprevious(pi)
    output_path.parent.mkdir(exist_ok=True, parents=True)
    etree.ElementTree(root).write(
        output_path, pretty_print=True, xml_declaration=True, encoding="UTF-8"
    )

## I. Parsing

First, set the static collection metadata as an `lxml.ElementTree`

In [50]:
catalog_header_semper: str = """<?xml version='1.0' encoding='UTF-8'?>
<?oxygen RNGSchema="schema.rng" type="xml"?>
<collection identifier="https://example.org/dts/collections/semper">
    <title>Digital Semper Edition (dev: collection)</title>
    <dublinCore>
        <creator>Gottfried Semper</creator>
        <language>de</language>
    </dublinCore>
    <!-- Add members -->
</collection>
"""

catalog_semper: ElementTree = etree.ElementTree(
    etree.fromstring(catalog_header_semper.encode("utf-8"))
)
print(etree.tostring(catalog_semper.getroot(), pretty_print=True, encoding="unicode"))

<collection identifier="https://example.org/dts/collections/semper">
    <title>Digital Semper Edition (dev: collection)</title>
    <dublinCore>
        <creator>Gottfried Semper</creator>
        <language>de</language>
    </dublinCore>
    <!-- Add members -->
</collection>



In [51]:
def parse_teifolder_to_dapytains(
    source: Path, target: Path, catalog: ElementTree
) -> etree.Element:
    def extract_volume_titles(etree):
        volume = etree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
            namespaces=NS,
        )

        monograph = etree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title[@level='m']",
            namespaces=NS,
        )

        titles = []
        if volume:
            titles.append(volume[0].text.strip())
        if monograph:
            titles.append(monograph[0].text.strip())
        return titles

    # Flow control
    if not source.exists():
        raise ValueError(f"Source directory does not exist: {source}")
    if not source.is_dir():
        raise ValueError(f"Source path is not a directory: {source}")

    ## Get the root node to retrieve the collection base_identifier
    catalog_root = catalog.getroot()
    base_identifier: str = catalog_root.get("identifier")
    if not base_identifier:
        raise ValueError("Catalog does not have an identifier attribute")

    ## (1) ROOT CATALOG - Make reference to Collection file in the catalog as members
    collection_xml_relative_path = f"./collection-{source.name}.xml"
    collection_catalog = etree.Element(
        "collection",
        identifier=f"{base_identifier}/{source.name}",
        filepath=str(collection_xml_relative_path),
    )

    ## (2) NEW 'Catalog - Manuscript as a Collection'
    collection_manuscript = etree.Element(
        "collection", identifier=f"{base_identifier}/{source.name}"
    )
    title = etree.SubElement(collection_manuscript, "title")
    first_file_etree = parse_xml(next(source.glob("*.xml")))
    extracted_title = first_file_etree.xpath(
        "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title", namespaces=NS
    )
    title.text = extracted_title[0].text.strip()
    members_collection_manuscript = etree.SubElement(collection_manuscript, "members")

    for source_tei_file in sorted(source.glob("*.xml")):
        source_tei_tree = parse_xml(source_tei_file)
        print(extract_volume_titles(etree=source_tei_tree))
        resource_identifier = f"{base_identifier}/{source.name}/{source_tei_file.stem}"
        print(resource_identifier)

        resource_xml_path = Path(target, "tei", source.name, source_tei_file.name)
        resource_xml_relative_path = f"../tei/{source.name}/{source_tei_file.name}"
        resource = etree.SubElement(
            members_collection_manuscript,
            "resource",
            identifier=resource_identifier,
            filepath=resource_xml_relative_path,
        )
        title = etree.SubElement(resource, "title")
        extracted_source_tei_title = source_tei_tree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title", namespaces=NS
        )[0]
        title.text = "".join(extracted_source_tei_title.itertext())

        write_xml_with_schema(source_tei_tree, resource_xml_path)

    ## (2.2) Write the 'Catalog - Manuscript as a Collection'
    collection_target_path = Path(target, "catalog", f"collection-{source.name}.xml")
    collection_manuscript_tree = etree.ElementTree(collection_manuscript)
    write_xml_with_schema(collection_manuscript_tree, collection_target_path)

    ## (3) Copy local dependencies (.dtd, .rng)
    for f in source.glob("*"):
        if f.is_file() and f.suffix in [".dtd", ".rng"]:
            shutil.copy(src=f, dst=Path(target, "tei", source.name, f.name))

    return collection_catalog

In [52]:
def append_collection_as_member_to_catalog(
    catalog_tree: ElementTree, collection: Element
) -> ElementTree:
    root = catalog_tree.getroot()
    members_element = root.find("members")
    if members_element is None:
        members_element = etree.SubElement(root, "members")
    members_element.append(collection)
    return catalog_tree

In [53]:
for source_dir in sorted(DIR_SEMPER_SAMPLE.iterdir()):
    if not source_dir.is_dir():
        continue
    print(f"Processing {source_dir.name}...")
    cm = parse_teifolder_to_dapytains(
        source=source_dir, catalog=catalog_semper, target=DIR_OUT
    )
    catalog_semper = append_collection_as_member_to_catalog(
        catalog_tree=catalog_semper, collection=cm
    )

print("\nFinal catalog:")
print(etree.tostring(catalog_semper.getroot(), pretty_print=True, encoding="unicode"))
write_xml_with_schema(
    tree=catalog_semper, output_path=Path(DIR_OUT, "catalog", "catalog.xml")
)

Processing 184...
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/184/20_Ms_184_1
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/184/20_Ms_184_2
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/184/20_Ms_184_3
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/184/20_Ms_184_4
Processing 185...
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/185/20_Ms_185_1
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/185/20_Ms_185_2
['Gottfried Semper: Der Stil. Kritische und kommentierte Ausgabe: Autograph:']
https://example.org/dts/collections/semper/185/20_Ms_185_

# WIPs

In [ ]:
# def parse_flat_teifolder_to_dapytains(
#     source: Path,
#     target: Path,
#     catalog: ElementTree,
#     etree: Element | None = None,
# ):  # -> etree.Element:
#     # Flow control
#     if not source.exists():
#         raise ValueError(f"Source directory does not exist: {source}")
#     if not source.is_dir():
#         raise ValueError(f"Source path is not a directory: {source}")
#     if not target.exists():
#         raise ValueError(f"Target directory does not exist: {target}")
#     if not target.is_dir():
#         raise ValueError(f"Target path is not a directory: {target}")
#     # if not (("catalog" in [stem for stem in list(target.glob("*"))]) and ("tei" in [stem for stem in list(target.glob("*"))])):
#     #     raise ValueError(f"Init target folder with Dapytain' catalog, tei and catalog.xml: {list(target.glob("*"))}")
#     # if not list(source.glob("*.xml")): raise ValueError(f"No XML files found in directory: {source}")

#     # Get the root node, "An ElementTree is mainly a document wrapper around a tree with a root node."
#     if isinstance(catalog, ElementTree):
#         root = catalog.getroot()
#     else:
#         root = catalog  # avoid problems with _Element and so on

#     # ... to retrieve the base_identifier
#     base_identifier: str = root.get("identifier")
#     if not base_identifier:
#         raise ValueError("Catalog does not have an identifier attribute")

#     # (1) generate the member statement for the collection catalog as a new collection
#     collection_target_path = Path(target, "catalog", f"{source.name}.xml")
#     collection = etree.Element(
#         "collection", identifier=f"{base_identifier}/{source.name}"
#     )

#     # ... and its elements
#     title = etree.SubElement(collection, "title", filepath=collection_target_path)
#     title.text = f"Manuscript {source.name}"

#     # (2) write out the new collection at target path

#     return collection


# cm = parse_flat_teifolder_to_dapytains(
#     source=DIR_SEMPER_SAMPLE / "184", target=DIR_OUT, catalog=catalog_semper
# )
# print_etree(cm)

In [ ]:
# for source_dir in sorted(DIR_SEMPER_SAMPLE.iterdir()):
#     if not source_dir.is_dir():
#         continue
#     print(f"Processing {source_dir.name}...")
#     cm = parse_teifolder_to_dapytains(source=source_dir, catalog=catalog_semper, target=DIR_OUT)
#     catalog_semper = append_collection_as_member_to_catalog(catalog_tree=catalog_semper, collection=cm)

# print("\nFinal catalog:")
# print(etree.tostring(catalog_semper.getroot(), pretty_print=True, encoding='unicode'))
# write_xml_with_schema(tree=catalog_semper, output_path=Path(DIR_OUT, "catalog", "catalog.xml"))

In [ ]:
# def create_root_collection_catalog_hwgw(
#     root_dir: Path, root_tei_file: Path, base_identifier
# ) -> ElementTree:
#     """Generate the root collection catalog for a TEI File Folder"""
#     first_tei = parse_xml(root_tei_file)
#     collection_root = etree.Element("collection", identifier=base_identifier)

#     # Add the collection metadata to the catalog
#     title = etree.SubElement(collection_root, "title")
#     title.text = extract_root_series_title(first_tei)
#     dublin = etree.SubElement(collection_root, "dublinCore")
#     creator = etree.SubElement(dublin, "creator")
#     creator.text = "Heinrich Wölfflin"
#     language = etree.SubElement(dublin, "language")
#     language.text = "de"
#     members = etree.SubElement(collection_root, "members")

#     for folder in sorted(root_dir.iterdir()):
#         if folder.is_dir() and folder.name.startswith("s"):
#             etree.SubElement(members, "collection", filepath=f"./{folder.name}.xml")

#     return etree.ElementTree(collection_root)


# def create_root_collection_catalog_semper(
#     root_dir: Path, root_tei_file: Path, base_identifier
# ) -> ElementTree:
#     """Generate the root collection catalog for a TEI File Folder"""
#     first_tei = parse_xml(root_tei_file)
#     collection_root = etree.Element("collection", identifier=base_identifier)

#     # Add the collection metadata to the catalog
#     title = etree.SubElement(collection_root, "title")
#     title.text = extract_root_series_title(first_tei)
#     dublin = etree.SubElement(collection_root, "dublinCore")
#     creator = etree.SubElement(dublin, "creator")
#     creator.text = "Gottfried Semper"
#     language = etree.SubElement(dublin, "language")
#     language.text = "de"
#     members = etree.SubElement(collection_root, "members")

#     # # for folder in sorted(root_dir.iterdir()):
#     # #     if folder.is_dir() and folder.name.startswith("s"):
#     # #         etree.SubElement(
#     # #             members,
#     # #             "collection",
#     # #             filepath=f"./{folder.name}.xml"
#     # #         )

#     return etree.ElementTree(collection_root)


# # catalog_hwgw = create_root_collection_catalog_hwgw(
# #     root_dir=DIR_HWGW_SAMPLE,
# #     root_tei_file=root_hwgw_tei,
# #     base_identifier='Unk'
# # )

In [ ]:
# root_catalog_semper = """<?xml version='1.0' encoding='UTF-8'?>
# <?oxygen RNGSchema="schema.rng" type="xml"?>
# <collection identifier="https://example.org/dts/collections/semper">
#     <title>Collection Semper</title>
#     <dublinCore>
#         <creator>Gottfried Semper</creator>
#         <language>de</language>
#     </dublinCore>
#     <!-- Add members -->
# </collection>
# """


# def create_subcollection_from_flat_dir() -> ElementTree:
#     return etree.ElementTree(subcollection)


# def create_collection(base_identifier, source_dir, metadatas: str) -> ElementTree:
#     collection = etree.Element("collection", identifier=base_identifier)
#     collection_members = etree.SubElement(collection, "members")
#     for folder in sorted(source_dir.iterdir()):
#         if not folder.is_dir():
#             # print(f"Exists file {folder}")
#             continue

#         etree.SubElement(
#             collection_members, "collection", filepath=f"./{folder.name}.xml"
#         )
#         for f in sorted(folder.iterdir()):
#             if not f.suffix == ".xml":
#                 # print(f"Skipping - {f}")
#                 continue

#     return etree.ElementTree(collection)


# c = create_collection_semper(
#     base_identifier="semper-digital-editions",
#     source_dir=DIR_SEMPER_SAMPLE,
#     metadatas=root_catalog_semper,
# )

In [ ]:
# c.tostring()